# gramlab — map stage (CPU): the grammar meaning map vs bag of words vs BM25 vs the teacher

Input: the **gramlab** notebook output (Version 3: `work/teacher/*`, `work/bm25`, `data/stsb.jsonl`, `results/*`).
**Accelerator: None** (no GPU quota), Internet on. *Save & Run All*; ≈ 40–60 min, almost all of it POS-tagging 659k passages.

The map: one 32-dim code per word, summed per grammar slot (SUBJ / VERB / OBJ / PP / ADJP / ADVP / OTHER, found by POS patterns —
no parser at inference), a linear readout onto the teacher's 384-dim space, trained by cosine regression on 191k parsed AMALGUM
sentences. The control is the identical model with one slot (bag of words). Scored on STS-B (teacher: 0.824) and on the retrieval
scoreboard (BM25 body-sentence recall@1 0.233, teacher ≈0.33), plus a BM25 + map hybrid.

In [ ]:
import os, subprocess, sys, time, glob, shutil, json
T0 = time.time()
def sh(cmd, must=True):
    print('\n$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if must and r.returncode != 0:
        raise SystemExit(f'command failed ({r.returncode}): {cmd}')
os.chdir('/kaggle/working')
if not os.path.exists('gramlab'):
    sh('git clone --depth 1 https://github.com/furevych/gramlab.git')
os.chdir('/kaggle/working/gramlab')
src = sorted(glob.glob('/kaggle/input/**/work/teacher/amalgum_sent.npy', recursive=True))
if not src:
    raise SystemExit('Attach the gramlab notebook output (Version 3) as an input')
root = src[0].split('/work/')[0]; print('input:', root)
os.makedirs('work', exist_ok=True); os.makedirs('results', exist_ok=True); os.makedirs('data', exist_ok=True)
for d in ['teacher', 'bm25']:
    if not os.path.exists(f'work/{d}'):
        shutil.copytree(f'{root}/work/{d}', f'work/{d}')
for f in glob.glob(f'{root}/results/*.json') + glob.glob(f'{root}/data/stsb.jsonl'):
    shutil.copy(f, 'results/' if f.endswith('.json') and '/results/' in f else 'data/')
try:
    import spacy; spacy.load('en_core_web_sm')
except Exception:
    sh('pip install -q spacy && python -m spacy download -q en_core_web_sm', must=False)
print(f'elapsed {(time.time()-T0)/60:.0f} min')

## Train both maps, score STS-B and retrieval

In [ ]:
sh('python scripts/map_stage.py --teacher work/teacher --index work/bm25 --queries work/bm25/eval_queries.jsonl '
   '--stsb data/stsb.jsonl --out work/map --results results/map.json --epochs 8 --n_process 4')
print(json.dumps(json.load(open('results/map.json')), indent=1)[:3000])
print(f'elapsed {(time.time()-T0)/60:.0f} min')

## What the slots learned: nearest words by code, and which slot each word's code weighs most

In [ ]:
import torch, numpy as np
sys.path.insert(0, '/kaggle/working/gramlab')
from gramlab.map import MapEncoder, SLOTS
enc = MapEncoder('work/map/slots')
codes = enc.model.code.weight.detach().numpy()[:len(enc.vocab.words)]
codes_n = codes / (np.linalg.norm(codes, axis=1, keepdims=True) + 1e-9)
def nearest(word, k=8):
    i = enc.vocab.index.get(word)
    if i is None: return []
    s = codes_n @ codes_n[i]; order = np.argsort(-s)[1:k+1]
    return [(enc.vocab.words[j], round(float(s[j]), 2)) for j in order]
for w in ['city', 'river', 'king', 'kill', 'hate', 'love', 'quickly', 'france', 'capital', 'born']:
    print(f'{w:10}', nearest(w))
gate = enc.model.slot_gate.detach().numpy()
print('\nslot gate norms (how much each slot scales a code):', {s: round(float(np.linalg.norm(gate[i])), 2) for i, s in enumerate(SLOTS)})

In [ ]:
shutil.rmtree('/kaggle/working/gramlab/.git', ignore_errors=True)
os.system('du -sh /kaggle/working/gramlab/work/* /kaggle/working/gramlab/results 2>/dev/null')
print(f'total {(time.time()-T0)/60:.0f} min')